In [12]:
# ---- cleaning notebook ----- #

# packages
import pickle
import numpy as np
import pandas as pd
import os

# scripts
import scripts.helper_functions as func

# what data set are we preprocessing
dataset = 'exp1'

# paths
raw_path = '~/code/ast_human/data/raw/' + dataset
clean_path = '~/code/ast_human/data/clean/' + dataset
datalist_path = '~/code/ast_human/data/datalists/' + dataset

# make full path if dirs don't exist
paths = [raw_path, clean_path, datalist_path]
for p in paths:
    os.makedirs(p, exist_ok=True)

# read data
raw_data_1 = pd.read_csv(raw_path + '/data_exp_264643-v3_task-1rv7.csv')
raw_data_2 = pd.read_csv(raw_path + '/data_exp_264643-v3_task-h671.csv')
raw_data = pd.concat([raw_data_1,raw_data_2])

print('csv loaded, N = ' + str(len(raw_data['Participant Public ID'].unique())))

/tmp/ipykernel_31243/499722819.py:26: DtypeWarning: Columns (0: Event Index) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data_1 = pd.read_csv(raw_path + '/data_exp_264643-v3_task-1rv7.csv')


csv loaded, N = 1493


/tmp/ipykernel_31243/499722819.py:27: DtypeWarning: Columns (0: Event Index) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data_2 = pd.read_csv(raw_path + '/data_exp_264643-v3_task-h671.csv')


In [13]:
# ----- basic checks ----- #

# any subjects go over the limit?
over = raw_data[raw_data["n_trials_stage"] > 50]
if True in over:   
    over = over['Participant Public ID'].unique()
    print('Check these for exceeding trial limits: ' + over)
else:
    print('No trial limits exceeded')

No trial limits exceeded


In [11]:
# rename some colums
raw_data = raw_data.rename(columns={'feedback_perceived': 'feedback', 'Manipulation: initial_dimension': 'initial_dimension'})

# remove ppts who failed checks
#raw_data = funs.remove_failures(raw_data)

# remove anyone who didn't pass CD
ids = pd.DataFrame(raw_data.groupby(['Participant Public ID'])['stages_passed'].max())
ids = ids[ids['stages_passed'] > 2].reset_index()

clean_data = raw_data[raw_data['Participant Public ID'].isin(ids['Participant Public ID'])]
clean_data.to_csv(clean_path + '/clean_data.csv')

In [16]:
# now let's make the datalists for modelling

ids = pd.DataFrame(raw_data.groupby(['Participant Public ID'])['stages_passed'].max())
ids = ids[ids['stages_passed'] > 5].reset_index()
raw_data = raw_data[raw_data['Participant Public ID'].isin(ids['Participant Public ID'])]
raw_data.to_csv(clean_path + 'pilot/extended_r1.csv')
mask = raw_data['feedback_perceived']

# run function on each dataset
#raw_data = raw_data[raw_data['stage']!=7]
subjs = raw_data['Participant Public ID'].unique()
subjs = subjs[~pd.isnull(subjs)]
datalist_test = [funs.make_datalist(raw_data, subj) for subj in subjs]

# save datalist
with open(datalist_path + 'pilot/extended2_r1_datalist.pkl', 'wb') as file:
    pickle.dump(datalist_test, file)
ids.to_csv(datalist_path + 'pilot/extended2_r1_datalist_ids.csv')